# 갑상선암 검진 정책 변화 분석 (학생용 Colab)

이 노트북은 **저장소 경로와 파일명이 이미 고정**되어 있어, 
**GitHub → Colab에서 열어 `런타임 > 모두 실행`** 하면 분석이 진행되도록 만든 버전입니다.

## 출력물
- 전체/남녀 연령표준화발생률 추세
- 2012=100 충격지수
- 버터플라이 3단 비교
- 연령 heatmap
- 감소율 / rebound 워터폴
- 버블차트
- ITS 분석
- 2014 권고안 기준 ITS 추가 분석
- 성별 × 연령별 rebound 분석
- 전체 rebound 기여 연령군 분석
- 24개 암종 비교 분석
- 자동 해석 요약


In [ ]:
import os, sys, subprocess, urllib.request, shutil, warnings
from urllib.parse import quote

warnings.filterwarnings('ignore')

print('환경 설정 중...')
subprocess.run(['apt-get', '-qq', 'install', 'fonts-nanum'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pandas', 'numpy', 'matplotlib', 'statsmodels', 'openpyxl', 'scipy'
], check=True)
print('환경 설정 완료')


In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')


In [ ]:
# =========================
# 저장소 고정 설정
# =========================
GITHUB_USER = 'runmc3812'
REPO_NAME = 'Healthcare_Informatics_Integration-2026-'
BRANCH = 'main'
BASE_PATH = '1조'

DATA_DIR = 'data'
SCRIPT_DIR = 'src'

MAIN_FILE = '국립암센터_24개종 암발생률_20260120.csv'
AGE_FILE = '24개_암종_성_연령_5세_별_암발생자수__발생률_20260324142549.csv'
SCRIPT_FILE = 'thyroid_colab_analysis.py'

OUTDIR = '/content/thyroid_analysis_outputs'
os.makedirs(OUTDIR, exist_ok=True)

def gh_raw_url(*parts):
    encoded = '/'.join(quote(str(p), safe='') for p in parts)
    return f'https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{encoded}'

main_source = gh_raw_url(BASE_PATH, DATA_DIR, MAIN_FILE)
age_source = gh_raw_url(BASE_PATH, DATA_DIR, AGE_FILE)
script_source = gh_raw_url(BASE_PATH, SCRIPT_DIR, SCRIPT_FILE)

print('main_source =', main_source)
print('age_source =', age_source)
print('script_source =', script_source)


In [ ]:
# 원본 파일 다운로드(추가 분석용)
local_main = '/content/main_thyroid_24cancers.csv'
local_age = '/content/age_thyroid_by_5y.csv'
local_script = '/content/thyroid_colab_analysis.py'

urllib.request.urlretrieve(main_source, local_main)
urllib.request.urlretrieve(age_source, local_age)
urllib.request.urlretrieve(script_source, local_script)

sys.path.append('/content')

from thyroid_colab_analysis import run_analysis

print('데이터 및 분석 스크립트 다운로드 완료')


In [ ]:
def read_csv_flex(path):
    for enc in ('utf-8', 'cp949', 'euc-kr'):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path)

def load_age_long(path):
    raw = read_csv_flex(path)

    if str(raw.iloc[0, 0]).strip() == '24개 암종별':
        raw = raw.iloc[1:].copy()

    new_cols = ['암종', '성별', '연령별']
    for c in raw.columns[3:]:
        year = str(c).split('.')[0]
        suffix = '발생자수' if not str(c).endswith('.1') else '조발생률'
        new_cols.append(f'{year}_{suffix}')
    raw.columns = new_cols

    raw = raw[raw['암종'].astype(str).str.contains('갑상선', na=False)].copy()

    value_cols = [c for c in raw.columns if '_' in c and c.split('_')[0].isdigit()]
    long_df = raw.melt(
        id_vars=['암종', '성별', '연령별'],
        value_vars=value_cols,
        var_name='year_metric',
        value_name='value'
    )

    long_df[['연도', '지표']] = long_df['year_metric'].str.extract(r'(\d+)_(.+)')
    long_df['연도'] = long_df['연도'].astype(int)
    long_df['value'] = pd.to_numeric(long_df['value'].replace('-', np.nan), errors='coerce')

    return long_df.drop(columns=['year_metric'])

def save_df(df, name):
    csv_path = os.path.join(OUTDIR, f'{name}.csv')
    xlsx_path = os.path.join(OUTDIR, f'{name}.xlsx')
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    df.to_excel(xlsx_path, index=False)
    return csv_path, xlsx_path

def its_table_and_plot(df_main, intervention_year, out_png):
    d = df_main[(df_main['암종'] == '갑상선') & (df_main['성별'] == '남녀전체')].copy()
    d = d.sort_values('발생연도')[['발생연도', '연령표준화발생률']].rename(columns={'연령표준화발생률': 'ASR'})

    d['time'] = np.arange(len(d))
    d['post'] = (d['발생연도'] >= intervention_year).astype(int)
    d['time_after'] = np.where(d['발생연도'] >= intervention_year, d['발생연도'] - intervention_year + 1, 0)

    X = sm.add_constant(d[['time', 'post', 'time_after']])
    model = sm.OLS(d['ASR'], X).fit()
    d['pred'] = model.predict(X)

    plt.figure(figsize=(10, 5))
    plt.plot(d['발생연도'], d['ASR'], marker='o', label='Observed ASR')
    plt.plot(d['발생연도'], d['pred'], linestyle='--', label=f'ITS fitted ({intervention_year})')
    plt.axvline(intervention_year, linestyle=':', linewidth=2, label=f'Intervention {intervention_year}')
    plt.title(f'갑상선암 ITS 분석 ({intervention_year} 권고안 기준)')
    plt.xlabel('연도')
    plt.ylabel('연령표준화발생률')
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200, bbox_inches='tight')
    plt.close()

    coef_df = pd.DataFrame({
        '변수': ['절편', '시간추세', '개입직후 수준변화', '개입후 기울기변화'],
        '계수': model.params.values,
        'p값': model.pvalues.values
    })
    return d, coef_df

def sex_age_rebound(long_df):
    df = long_df[
        (long_df['지표'] == '조발생률') &
        (long_df['성별'].isin(['남자', '여자'])) &
        (~long_df['연령별'].isin(['계', '연령미상']))
    ].copy()

    wide = df.pivot_table(index=['성별', '연령별'], columns='연도', values='value', aggfunc='first')
    for y in [2015, 2023]:
        if y not in wide.columns:
            wide[y] = np.nan

    out = wide[[2015, 2023]].copy()
    out.columns = ['2015 조발생률', '2023 조발생률']
    out['2015→2023 변화율(%)'] = (out['2023 조발생률'] / out['2015 조발생률'] - 1) * 100
    out = out.replace([np.inf, -np.inf], np.nan).reset_index()

    top = out.sort_values('2015→2023 변화율(%)', ascending=False).head(12)
    labels = top['성별'] + ' / ' + top['연령별']
    plt.figure(figsize=(10, 6))
    plt.barh(labels[::-1], top['2015→2023 변화율(%)'].iloc[::-1])
    plt.title('성별 × 연령별 갑상선암 Rebound 상위군 (2015→2023)')
    plt.xlabel('변화율(%)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, 'sex_age_rebound_top12.png'), dpi=200, bbox_inches='tight')
    plt.close()
    return out

def rebound_contribution(long_df):
    df = long_df[
        (long_df['지표'] == '발생자수') &
        (long_df['성별'] == '계') &
        (~long_df['연령별'].isin(['계', '연령미상']))
    ].copy()

    wide = df.pivot_table(index='연령별', columns='연도', values='value', aggfunc='first')
    for y in [2015, 2023]:
        if y not in wide.columns:
            wide[y] = np.nan

    out = wide[[2015, 2023]].copy()
    out.columns = ['2015 발생자수', '2023 발생자수']
    out['증가분'] = out['2023 발생자수'] - out['2015 발생자수']
    out = out.sort_values('증가분', ascending=False).reset_index()

    pos = out[out['증가분'] > 0].copy()
    if len(pos):
        pos['기여도(%)'] = pos['증가분'] / pos['증가분'].sum() * 100

        plt.figure(figsize=(10, 6))
        top = pos.head(12)
        plt.barh(top['연령별'].iloc[::-1], top['증가분'].iloc[::-1])
        plt.title('2015→2023 전체 Rebound 기여 연령군 (증가분 기준)')
        plt.xlabel('발생자수 증가분')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTDIR, 'rebound_contribution_top12.png'), dpi=200, bbox_inches='tight')
        plt.close()
    return out

def cancer_benchmark(df_main):
    df = df_main[df_main['성별'] == '남녀전체'].copy()
    wide = df.pivot_table(index='암종', columns='발생연도', values='연령표준화발생률', aggfunc='first')
    for y in [2012, 2015, 2023]:
        if y not in wide.columns:
            wide[y] = np.nan

    out = wide[[2012, 2015, 2023]].copy()
    out.columns = ['2012 ASR', '2015 ASR', '2023 ASR']
    out['2012→2015 변화율(%)'] = (out['2015 ASR'] / out['2012 ASR'] - 1) * 100
    out['2015→2023 변화율(%)'] = (out['2023 ASR'] / out['2015 ASR'] - 1) * 100
    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index()

    rebound_top = out.sort_values('2015→2023 변화율(%)', ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    plt.barh(rebound_top['암종'].iloc[::-1], rebound_top['2015→2023 변화율(%)'].iloc[::-1])
    plt.title('24개 암종 중 2015→2023 Rebound 상위 10개')
    plt.xlabel('변화율(%)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, 'cancer_rebound_top10.png'), dpi=200, bbox_inches='tight')
    plt.close()

    return out

print('보조 함수 정의 완료')


## 1. 기존 분석 실행

In [ ]:
print('\n기존 분석 실행 중...')
results = run_analysis(main_source=main_source, age_source=age_source, outdir=OUTDIR)

display(Markdown('## 핵심 요약표'))
display(results['summary_change'])

display(Markdown('## 2012→2015 감소폭 상위 10개 연령군'))
display(results['age_change_total'].sort_values('2012→2015 변화율(%)').head(10))

display(Markdown('## 2015→2023 rebound 상위 10개 연령군'))
display(results['age_change_total'].sort_values('2015→2023 변화율(%)', ascending=False).head(10))


## 2. 추가 분석용 데이터 로드

In [ ]:
main_df = read_csv_flex(local_main)
age_long = load_age_long(local_age)

print('main_df shape =', main_df.shape)
print('age_long shape =', age_long.shape)
display(age_long.head())


## 3. 2014 권고안 기준 ITS

In [ ]:
its_series_2014, its_coef_2014 = its_table_and_plot(
    main_df,
    intervention_year=2014,
    out_png=os.path.join(OUTDIR, 'its_2014_policy.png')
)
save_df(its_coef_2014, 'its_2014_coefficients')

display(Markdown('## 추가분석 1: 2014 권고안 기준 ITS 계수'))
display(its_coef_2014)
display(Markdown(f"- 그림 저장: `{os.path.join(OUTDIR, 'its_2014_policy.png')}`"))


## 4. 성별 × 연령별 rebound

In [ ]:
sex_age_df = sex_age_rebound(age_long)
save_df(sex_age_df, 'sex_age_rebound')

display(Markdown('## 추가분석 2: 성별×연령별 rebound 상위 10개'))
display(sex_age_df.sort_values('2015→2023 변화율(%)', ascending=False).head(10))


## 5. 전체 rebound 기여 연령군

In [ ]:
contrib_df = rebound_contribution(age_long)
save_df(contrib_df, 'rebound_contribution')

display(Markdown('## 추가분석 3: 전체 rebound 기여 연령군 상위 10개'))
display(contrib_df.head(10))


## 6. 24개 암종 비교

In [ ]:
benchmark_df = cancer_benchmark(main_df)
save_df(benchmark_df, 'cancer_benchmark_24')

display(Markdown('## 추가분석 4: 24개 암종 중 rebound 상위 10개'))
display(benchmark_df.sort_values('2015→2023 변화율(%)', ascending=False).head(10))


## 7. 자동 요약 생성

In [ ]:
summary_path = os.path.join(OUTDIR, 'auto_summary.txt')
extra_summary_path = os.path.join(OUTDIR, 'auto_summary_additional.txt')

extra_lines = []
try:
    top_rebound = sex_age_df.sort_values('2015→2023 변화율(%)', ascending=False).head(5)
    top_contrib = contrib_df.head(5)
    thyroid_rank = benchmark_df.sort_values('2015→2023 변화율(%)', ascending=False).reset_index(drop=True)
    thyroid_rank_num = thyroid_rank.index[thyroid_rank['암종'] == '갑상선'].tolist()
    thyroid_rank_num = thyroid_rank_num[0] + 1 if thyroid_rank_num else None

    extra_lines.append('[추가 요약]')
    extra_lines.append('- 2014년 권고안 기준 ITS를 별도로 생성하여 정책 분기점을 직접 점검했습니다.')
    extra_lines.append('- 성별×연령 분석에서는 청년층(특히 20~34세)에서 rebound가 가장 두드러지는지 확인할 수 있습니다.')
    extra_lines.append("- 2015→2023 증가분 기여 분석으로 '변화율이 큰 연령'과 '실제 전체 증가를 많이 만든 연령'을 구분했습니다.")
    if thyroid_rank_num is not None:
        extra_lines.append(f'- 24개 암종 비교에서 갑상선암의 2015→2023 rebound 순위는 {thyroid_rank_num}위입니다.')
    extra_lines.append('')
    extra_lines.append('[성별×연령 rebound 상위 5개]')
    for _, r in top_rebound.iterrows():
        extra_lines.append(f"- {r['성별']} {r['연령별']}: {r['2015→2023 변화율(%)']:.1f}%")
    extra_lines.append('')
    extra_lines.append('[전체 rebound 기여 상위 5개]')
    for _, r in top_contrib.iterrows():
        if pd.notna(r['증가분']):
            extra_lines.append(f"- {r['연령별']}: 증가분 {int(r['증가분'])}명")
except Exception as e:
    extra_lines.append(f'[추가 요약 생성 중 오류] {e}')

with open(extra_summary_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(extra_lines))

if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as f:
        print('\n[기존 자동 요약]\n')
        print(f.read())

if os.path.exists(extra_summary_path):
    with open(extra_summary_path, 'r', encoding='utf-8') as f:
        print('\n' + f.read())


## 8. 결과 압축 및 다운로드

In [ ]:
zip_path = shutil.make_archive('/content/thyroid_analysis_outputs', 'zip', OUTDIR)
print('\n압축 완료:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('자동 다운로드는 Colab에서만 동작합니다:', e)

print('\n완료: GitHub의 이 노트북을 Colab에서 열어 `모두 실행`하면 전체 분석이 생성됩니다.')
